# 00 — Main results: mean ± std trên nhiều seed

Notebook canonical cho main results: chạy mọi method trên `SEEDS`, đọc final-test record của từng run và xuất bảng paper-ready dạng **mean ± sample std**.

- Mặc định: seeds `42, 43, 44`, corpus `100k`, 5 epochs.
- `SEEDS` một phần tử vẫn chạy được (smoke run / thử một setting): sample std cần ít nhất hai seed, nên bảng khi đó in mean trần, không in `± nan`. Cột `_n` luôn nói có mấy seed đứng sau mỗi ô.
- Mỗi run nằm tại `<RUN_ROOT>/<method>/seed_<seed>` nên có thể resume an toàn.
- Teacher cache dùng chung giữa method và seed.
- Bảng cuối nhân điểm với 100 để khớp `tables/main_results.tex`; CSV raw vẫn giữ thang `[0, 1]`.
- `std` là sample standard deviation (`ddof=1`) trên đúng số seed của `SEEDS`.
- `avg_all` chỉ trung bình 9 benchmark sentence-level; 5 retrieval benchmark chỉ đi vào `avg_retrieval`.


In [19]:
# 1. Cấu hình thí nghiệm
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"

PAIRS = {
    "qwen3_0.6b_to_minilm_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 12,
    },
    "bge_m3_to_minilm_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "teacher_pooling": "cls",
        "teacher_special_token": "▁",
        "emo_teacher_special_token": "<s>",
        "min_vram_gib": 12,
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "teacher_special_token": "Ġ",
        "emo_teacher_special_token": None,
        "min_vram_gib": 24,
    },
}
PAIR = "qwen3_0.6b_to_minilm_h384"

DATASETS = {
    "talas_15k": {
        "path": Path("data/train_set/merged_3_data_5k_each.csv"),
        "build": None,
    },
    "100k": {
        "path": Path("data/train_set/train_100k.csv"),
        "build": None,
    },
    "150k": {
        "path": Path("data/train_set/train_150k.csv"),
        "build": "scripts/data/build_train_corpus.py --total 150000",
    },
    "200k": {
        "path": Path("data/train_set/train_200k.csv"),
        "build": "scripts/data/build_train_corpus.py --total 200000",
    },
}
DATASET = "talas_15k"
SEEDS = [42]

MAX_LENGTH = 256
EPOCHS = 5
NUM_WORKERS = 2
CUDA_VISIBLE_DEVICES = "0,1"
STOP_ON_ERROR = True
REQUIRE_ALL_SEEDS = True
# False = một lần eval duy nhất, thẳng trên test split (mặc định của main.py).
# True = giao thức held-out: eval validation mỗi epoch + eval test cuối run (2 lượt).
HOLD_OUT_VALIDATION = False
EVAL_EVERY = 0
EVAL_RETRIEVAL = True
AUTO_FETCH_DATA = True
SAVE_TO_GOOGLE_DRIVE = False

# Tất cả learned baselines xuất hiện trong main-results table, cộng method của bài.
# Cùng batch size làm cho ms/step và samples/s trong efficiency table so sánh trực tiếp.
METHOD_SETTINGS = {
    # "rkd":     {"batch_size": 128, "learning_rate": 7e-5},
    # "stella":  {"batch_size": 128, "learning_rate": 5e-5},
    # "cdm":     {"batch_size": 128, "learning_rate": 2e-5},
    # "dskd":    {"batch_size": 128, "learning_rate": 2e-5},
    # "emo":     {"batch_size": 128, "learning_rate": 1e-5},
    "talas":   {"batch_size": 128, "learning_rate": 2e-5},
    "geoode":  {"batch_size": 128, "learning_rate": 7e-5},
}

# Đám mây điểm mà L_topo đọc, tách khỏi batch của optimizer: 0 = một diagram cho
# cả batch (hành vi cũ), b >= 2 cắt mỗi batch thành batch_size // b đám mây rời
# nhau rồi lấy trung bình loss. Chính b quyết định thang lọc: L_H0 so đúng b - 1
# death time, nên nó là thang hình học chứ không phải nút chỉnh bộ nhớ.
GEOODE_H0_BATCH_SIZE = 128

PAIR_CONFIG = PAIRS[PAIR]
DATASET_CONFIG = DATASETS[DATASET]
TEACHER_MODEL = PAIR_CONFIG["teacher"]
STUDENT_MODEL = PAIR_CONFIG["student"]
TEACHER_POOLING = PAIR_CONFIG["teacher_pooling"]
TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["teacher_special_token"]
EMO_TEACHER_SPECIAL_TOKEN = PAIR_CONFIG["emo_teacher_special_token"]
TRAIN_DATA_REL = DATASET_CONFIG["path"]
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
# Điền tên run cũ để resume sau khi restart runtime; None tạo run mới.
RUN_NAME_OVERRIDE = None
RUN_NAME = RUN_NAME_OVERRIDE or f"{PAIR}_{DATASET}_{len(SEEDS)}seeds_all_methods_{RUN_STAMP}"

# Một seed là hợp lệ (smoke run, hoặc thử một setting trước khi trả tiền cho ba
# seed); chỉ bảng mean ± std là mất phần std, và cell 6 xử lý đúng chuyện đó.
assert SEEDS and len(SEEDS) == len(set(SEEDS)), f"SEEDS phải không rỗng và không trùng: {SEEDS}"
print(f"Pair: {PAIR}")
print(f"Dataset: {DATASET}; seeds: {SEEDS}")
print(f"Run: {RUN_NAME}")
print(f"Methods ({len(METHOD_SETTINGS)}): {', '.join(METHOD_SETTINGS)}")


Pair: qwen3_0.6b_to_minilm_h384
Dataset: talas_15k; seeds: [42]
Run: qwen3_0.6b_to_minilm_h384_talas_15k_1seeds_all_methods_20260904-121748
Methods (2): talas, geoode


In [20]:
# 2. Dùng repo hiện tại hoặc clone trên Colab; cài dependencies.
import subprocess
import sys

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if PROJECT_DIR.exists():
        assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

head_before = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
head_after = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git HEAD: {head_before} -> {head_after}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


Git HEAD: b4e6e8f -> b4e6e8f


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/embedding-kd/requirements.txt'], returncode=0)

In [21]:
# 3. Output, GPU và dữ liệu
import os
import torch

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
RETRIEVAL_DIR = PROJECT_DIR / "data" / "test_set" / "retrieval"

def run_script(argv, what):
    print(f"[data] {what}: python3 {' '.join(argv)}")
    subprocess.run([sys.executable, *argv], cwd=PROJECT_DIR, check=True)

for split in ("train_set", "val_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), f"Thiếu data: {split_dir}"

missing_retrieval = [
    name for name in ("arguana", "fiqa", "scidocs", "scifact", "nfcorpus")
    if not (RETRIEVAL_DIR / name / "corpus.csv").is_file()
]
if missing_retrieval and (EVAL_RETRIEVAL or DATASET_CONFIG["build"]):
    if not AUTO_FETCH_DATA:
        raise FileNotFoundError(f"Thiếu retrieval data: {missing_retrieval}")
    run_script(["scripts/data/download_retrieval_benchmarks.py"], "tải retrieval benchmarks")

if not TRAIN_DATA.is_file():
    build = DATASET_CONFIG["build"]
    if build is None or not AUTO_FETCH_DATA:
        raise FileNotFoundError(f"Thiếu training data: {TRAIN_DATA}")
    run_script(build.split(), f"dựng corpus {DATASET}")

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi chạy.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU phải hỗ trợ BF16.")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
largest_gib = 0.0
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    gib = props.total_memory / 2**30
    largest_gib = max(largest_gib, gib)
    print(f"cuda:{index}: {props.name} ({gib:.1f} GiB)")
if largest_gib < PAIR_CONFIG["min_vram_gib"]:
    print(f"[WARN] Nên có >= {PAIR_CONFIG['min_vram_gib']} GiB trên một GPU.")
print(f"Training data: {TRAIN_DATA}")
print(f"Output root: {RUN_ROOT}")

[data] tải retrieval benchmarks: python3 scripts/data/download_retrieval_benchmarks.py
cuda:0: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Training data: /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv
Output root: /content/embedding-kd/runs/qwen3_0.6b_to_minilm_h384_talas_15k_1seeds_all_methods_20260904-121748


In [22]:
# 4. Tạo plan: một job cho mỗi (method, seed).
import shlex

def build_command(name, settings, seed):
    method = settings.get("method", name)
    method_dir = RUN_ROOT / name / f"seed_{seed}"
    command = [
        sys.executable, str(PROJECT_DIR / "main.py"),
        "--method", method,
        "--train_data", str(TRAIN_DATA),
        "--student_model", STUDENT_MODEL,
        "--teacher_model", TEACHER_MODEL,
        "--teacher_pooling", TEACHER_POOLING,
        "--batch_size", str(settings["batch_size"]),
        "--epochs", str(EPOCHS),
        "--save_every", str(EPOCHS),
        "--lr", str(settings["learning_rate"]),
        "--max_length", str(MAX_LENGTH),
        "--save_dir", str(method_dir),
        "--num_workers", str(NUM_WORKERS),
        "--seed", str(seed),
        "--eval_every", str(EVAL_EVERY),
        "--no_wandb",
    ]
    if HOLD_OUT_VALIDATION:
        command.append("--no-evaluate_test_each_epoch")
    if not EVAL_RETRIEVAL:
        command.append("--no_eval_retrieval")
    if method == "cdm":
        command.extend(["--teacher_special_token", TEACHER_SPECIAL_TOKEN])
    if method == "emo" and EMO_TEACHER_SPECIAL_TOKEN is not None:
        command.extend(["--teacher_special_token", EMO_TEACHER_SPECIAL_TOKEN])
    if method in ("talas", "geoode", "rkd"):
        command.extend(["--cache_dir", str(CACHE_DIR)])
    if method == "geoode":
        command.extend([
            "--lambda_topo", "1.0",
            "--lambda_ctr", "0.0",
            "--gauge_refit_every", "1",
            "--topo_batch_size", str(GEOODE_H0_BATCH_SIZE),
        ])
    command.extend(settings.get("args", []))
    return command

JOBS = [
    {
        "method": name,
        "seed": seed,
        "output_dir": RUN_ROOT / name / f"seed_{seed}",
        "command": build_command(name, settings, seed),
    }
    for name, settings in METHOD_SETTINGS.items()
    for seed in SEEDS
]
print(f"Plan: {len(METHOD_SETTINGS)} methods × {len(SEEDS)} seeds = {len(JOBS)} jobs")
for job in JOBS:
    print(f"[{job['method'].upper()} seed={job['seed']}] {shlex.join(job['command'])}")
    print()


Plan: 2 methods × 1 seeds = 2 jobs
[TALAS seed=42] /usr/bin/python3 /content/embedding-kd/main.py --method talas --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --batch_size 128 --epochs 5 --save_every 5 --lr 2e-05 --max_length 256 --save_dir /content/embedding-kd/runs/qwen3_0.6b_to_minilm_h384_talas_15k_1seeds_all_methods_20260904-121748/talas/seed_42 --num_workers 2 --seed 42 --eval_every 0 --no_wandb --cache_dir /content/embedding-kd/runs/teacher_cache

[GEOODE seed=42] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --save_d

In [23]:
# 5. Chạy tuần tự, ghi full log và resume theo từng job hoàn tất.
import json
import time

PROGRESS_EVERY_SEC = 30
PROGRESS_MAX_CHARS = 160

def final_test_record(metrics_path):
    if not metrics_path.is_file():
        return None
    found = None
    with metrics_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            if record.get("test") and record.get("train") is None:
                found = record
    return found

def stream_output(stream, log_handle):
    buffer = ""
    last_progress = 0.0
    progress_shown = False
    while True:
        chunk = stream.read(4096)
        if not chunk:
            break
        buffer += chunk
        parts = buffer.replace("\r\n", "\n").replace("\r", "\n").split("\n")
        buffer = parts.pop()
        for line in parts:
            log_handle.write(line + "\n")
            if "%|" in line:
                now = time.perf_counter()
                if now - last_progress >= PROGRESS_EVERY_SEC:
                    print("\r" + line[:PROGRESS_MAX_CHARS].ljust(PROGRESS_MAX_CHARS), end="", flush=True)
                    last_progress = now
                    progress_shown = True
            elif line.strip():
                if progress_shown:
                    print()
                    progress_shown = False
                print(line)
        log_handle.flush()
    if buffer:
        log_handle.write(buffer + "\n")
        if "%|" not in buffer:
            print(buffer)
    if progress_shown:
        print()

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
env["TOKENIZERS_PARALLELISM"] = "false"
env["WANDB_MODE"] = "disabled"
env["TQDM_MININTERVAL"] = str(PROGRESS_EVERY_SEC)
run_status = []

for position, job in enumerate(JOBS, start=1):
    method = job["method"]
    seed = job["seed"]
    output_dir = job["output_dir"]
    metrics_path = output_dir / "metrics.jsonl"
    log_path = RUN_ROOT / method / f"seed_{seed}_train.log"
    if final_test_record(metrics_path) is not None:
        print(f"[SKIP] {method} seed={seed} đã có final test")
        run_status.append({"method": method, "seed": seed, "status": "skipped_complete", "seconds": 0.0})
        continue
    if metrics_path.exists():
        raise RuntimeError(
            f"Run dở dang: {metrics_path}. Xóa riêng seed này hoặc dùng RUN_NAME mới."
        )
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("\n" + "#" * 88)
    print(f"JOB {position}/{len(JOBS)}: {method.upper()} — seed {seed}")
    print(f"Log: {log_path}")
    print("#" * 88)
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            job["command"], cwd=PROJECT_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        assert process.stdout is not None
        stream_output(process.stdout, log_handle)
        return_code = process.wait()
    elapsed = time.perf_counter() - started
    status = "complete" if return_code == 0 and final_test_record(metrics_path) else "failed"
    run_status.append({"method": method, "seed": seed, "status": status, "seconds": elapsed})
    print(f"[{status.upper()}] {method} seed={seed} in {elapsed / 60:.1f} min")
    if status == "failed" and STOP_ON_ERROR:
        raise RuntimeError(f"Job failed; xem {log_path}")

print("\nRun status:")
for item in run_status:
    print(f"  {item['method']:8s} seed={item['seed']} {item['status']:18s} {item['seconds'] / 60:8.1f} min")



########################################################################################
JOB 1/2: TALAS — seed 42
Log: /content/embedding-kd/runs/qwen3_0.6b_to_minilm_h384_talas_15k_1seeds_all_methods_20260904-121748/talas/seed_42_train.log
########################################################################################
Configuration for TALAS method:
  task_type                 : pair_cls
  max_length                : 256
  batch_size                : 128
  epochs                    : 5
  learning_rate             : 2e-05
  min_lr                    : 2e-06
  warmup_ratio              : 0.06
  w_task                    : 0.001
  alpha_dtw                 : 0.5
  w_cls                     : 1.0
  temperature               : 0.1
  student_model_name        : nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
  teacher_model_name        : Qwen/Qwen3-Embedding-0.6B
  teacher_dtype             : bfloat16
  pooling_method            : last_token
  student_special_token     : ##
  t

In [18]:
# 6. Đọc final test của từng seed và tạo bảng mean ± sample std.
import numpy as np
import pandas as pd
from IPython.display import display

BENCHMARK_ORDER = [
    "banking77", "tweet", "emotion",
    "mrpc", "scitail", "wic",
    "sick", "sts12", "stsb",
]
SUMMARY_ORDER = ["avg_iod", "avg_ood", "avg_retrieval", "avg_all"]

def benchmark_name(path):
    name = Path(path).stem
    return name[:-5] if name.endswith("_test") else name

def score_from_payload(family, raw_values):
    if family == "classification":
        return float(raw_values["f1"])
    if family == "pair":
        return float(raw_values["average_precision"])
    if family == "sts":
        return float(raw_values)
    if family == "retrieval":
        return float(raw_values["ndcg_at_10"])
    raise KeyError(f"Unknown family: {family}")

rows = []
missing = []
for method in METHOD_SETTINGS:
    for seed in SEEDS:
        metrics_path = RUN_ROOT / method / f"seed_{seed}" / "metrics.jsonl"
        record = final_test_record(metrics_path)
        if record is None:
            missing.append((method, seed, str(metrics_path)))
            continue
        payload = record["test"]
        row = {"method": method, "seed": seed}
        for family in ("classification", "pair", "sts", "retrieval"):
            for path, values in payload.get(family, {}).items():
                row[benchmark_name(path)] = score_from_payload(family, values)
        summary = payload["summary"]
        for key in SUMMARY_ORDER:
            value = summary.get(key)
            row[key] = np.nan if value is None else float(value)
        # Tính lại từ raw score để cả run cũ cũng loại retrieval khỏi avg_all.
        row["avg_all"] = float(np.mean([row[name] for name in BENCHMARK_ORDER if name in row]))
        rows.append(row)

if missing:
    for method, seed, path in missing:
        print(f"[MISSING] {method} seed={seed}: {path}")
    if REQUIRE_ALL_SEEDS:
        raise RuntimeError(f"Thiếu {len(missing)} final-test runs; chưa aggregate để tránh bảng thiếu seed.")

by_seed = pd.DataFrame(rows)
assert not by_seed.empty, "Không tìm thấy final-test result."
metric_order = [name for name in BENCHMARK_ORDER + SUMMARY_ORDER if name in by_seed.columns]
by_seed = by_seed[["method", "seed", *metric_order]].sort_values(["method", "seed"])

counts = by_seed.groupby("method")["seed"].nunique()
if REQUIRE_ALL_SEEDS and not counts.eq(len(SEEDS)).all():
    raise RuntimeError(f"Seed counts không đủ:\n{counts}")

# Sample std (ddof=1) không tồn tại với một seed: pandas trả NaN, và
# f"{nan:.2f}" in ra "nan". Một seed thì in mean trần; cột _n vẫn nói n = 1.
def mean_pm_std(mean, std, separator=" ± ", digits=2):
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f}{separator}{std:.{digits}f}"

grouped = by_seed.groupby("method", sort=False)[metric_order]
means = grouped.mean()
stds = grouped.std(ddof=1)
ns = grouped.count()

wide_columns = {}
for metric in metric_order:
    wide_columns[f"{metric}_mean"] = means[metric]
    wide_columns[f"{metric}_std"] = stds[metric]
    wide_columns[f"{metric}_n"] = ns[metric]
mean_std_numeric = pd.DataFrame(wide_columns)

paper_metrics = [name for name in BENCHMARK_ORDER + ["avg_all"] if name in metric_order]
paper_display = pd.DataFrame(index=means.index)
paper_latex = pd.DataFrame(index=means.index)
for metric in paper_metrics:
    paper_display[metric] = [
        mean_pm_std(mean * 100, std * 100)
        for mean, std in zip(means[metric], stds[metric])
    ]
    paper_latex[metric] = [
        mean_pm_std(mean * 100, std * 100, " $\\pm$ ")
        for mean, std in zip(means[metric], stds[metric])
    ]

by_seed.to_csv(RUN_ROOT / "final_test_by_seed.csv", index=False)
mean_std_numeric.to_csv(RUN_ROOT / "final_test_mean_std.csv")
paper_display.to_csv(RUN_ROOT / "final_test_mean_std_paper.csv")
(RUN_ROOT / "final_test_mean_std.tex").write_text(
    paper_latex.to_latex(escape=False), encoding="utf-8"
)
pd.DataFrame(run_status).to_csv(RUN_ROOT / "run_status.csv", index=False)

print("FINAL TEST — EACH SEED (raw [0, 1])")
display(by_seed.style.format(precision=4))
label = "MEAN ± SAMPLE STD" if len(SEEDS) > 1 else "MEAN (một seed: không có std)"
print(f"FINAL TEST — {label} (paper scale [0, 100])")
display(paper_display)
print(f"Saved aggregate files to: {RUN_ROOT}")


FINAL TEST — EACH SEED (raw [0, 1])


,method,seed,banking77,tweet,emotion,mrpc,scitail,wic,sick,sts12,stsb,avg_iod,avg_ood,avg_retrieval,avg_all
0,geoode,42,0.9168,0.7460,0.6684,0.8506,0.8343,0.6749,0.7789,0.7214,0.7829,0.7087,0.8080,nan,0.7749
1,geoode,43,0.9159,0.7449,0.6726,0.8494,0.8347,0.6741,0.7794,0.7218,0.7841,0.7102,0.8077,nan,0.7752
2,geoode,44,0.9178,0.7419,0.6655,0.8500,0.8347,0.6744,0.7800,0.7218,0.7839,0.7079,0.8077,nan,0.7744


FINAL TEST — MEAN ± SAMPLE STD (paper scale [0, 100])


,banking77,tweet,emotion,mrpc,scitail,wic,sick,sts12,stsb,avg_all
method,,,,,,,,,,
geoode,91.68 ± 0.09,74.43 ± 0.21,66.88 ± 0.36,85.00 ± 0.06,83.46 ± 0.02,67.44 ± 0.04,77.94 ± 0.06,72.17 ± 0.02,78.36 ± 0.07,77.48 ± 0.04


Saved aggregate files to: /content/embedding-kd/runs/qwen3_0.6b_to_minilm_h384_100k_3seeds_all_methods_20260904-120525


In [ ]:
# 7. Main-results efficiency table — every baseline, measured from the same runs
import json

def read_jsonl(path):
    if not path.is_file():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

runtime_rows = []
for method in METHOD_SETTINGS:
    for seed in SEEDS:
        run_dir = RUN_ROOT / method / f"seed_{seed}"
        steps = pd.DataFrame(read_jsonl(run_dir / "step_metrics.jsonl"))
        timed = steps.query("step_seconds > 0 and global_step > 10") if not steps.empty else steps
        train_seconds = steps.loc[steps.get("step_seconds", 0) > 0, "step_seconds"].sum() if not steps.empty else np.nan
        epoch_records = read_jsonl(run_dir / "metrics.jsonl")
        peaks = [record["train"].get("peak_memory_mb", np.nan) for record in epoch_records if isinstance(record.get("train"), dict)]
        runtime_rows.append({
            "method": method,
            "seed": seed,
            "mean_step_ms": 1000 * timed["step_seconds"].mean() if not timed.empty else np.nan,
            "samples_per_second": timed["batch_size"].sum() / timed["step_seconds"].sum() if not timed.empty else np.nan,
            "train_gpu_minutes": train_seconds / 60,
            "peak_memory_gib": np.nanmax(peaks) / 1024 if peaks else np.nan,
        })

efficiency_by_seed = pd.DataFrame(runtime_rows).merge(
    by_seed[["method", "seed", "avg_all"]], on=["method", "seed"], how="left"
)
efficiency_by_seed.to_csv(RUN_ROOT / "efficiency_by_seed.csv", index=False)
required_runtime = ["mean_step_ms", "samples_per_second", "train_gpu_minutes", "peak_memory_gib"]
if REQUIRE_ALL_SEEDS and efficiency_by_seed[required_runtime].isna().any(axis=None):
    bad = efficiency_by_seed.loc[efficiency_by_seed[required_runtime].isna().any(axis=1), ["method", "seed"]]
    raise RuntimeError(f"Missing efficiency artifacts for:\n{bad.to_string(index=False)}")
efficiency_summary = efficiency_by_seed.groupby("method", sort=False).agg(
    avg_mean=("avg_all", "mean"), avg_sd=("avg_all", "std"),
    step_ms_mean=("mean_step_ms", "mean"), step_ms_sd=("mean_step_ms", "std"),
    throughput_mean=("samples_per_second", "mean"), throughput_sd=("samples_per_second", "std"),
    train_min_mean=("train_gpu_minutes", "mean"), train_min_sd=("train_gpu_minutes", "std"),
    memory_mean=("peak_memory_gib", "mean"), memory_sd=("peak_memory_gib", "std"),
    n=("avg_all", "count"),
).reset_index()

def mean_sd(mean, sd, digits=2):
    # Cùng quy ước với cell 6: một seed thì không có sample std để in.
    if pd.isna(sd):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {sd:.{digits}f}"

efficiency_table = pd.DataFrame({
    "Method": efficiency_summary.method.str.upper(),
    "AVG ↑": [mean_sd(100*m, 100*s) for m, s in zip(efficiency_summary.avg_mean, efficiency_summary.avg_sd)],
    "ms/step ↓": [mean_sd(m, s, 1) for m, s in zip(efficiency_summary.step_ms_mean, efficiency_summary.step_ms_sd)],
    "samples/s ↑": [mean_sd(m, s, 1) for m, s in zip(efficiency_summary.throughput_mean, efficiency_summary.throughput_sd)],
    "GPU train min ↓": [mean_sd(m, s, 1) for m, s in zip(efficiency_summary.train_min_mean, efficiency_summary.train_min_sd)],
    "peak GiB ↓": [mean_sd(m, s, 2) for m, s in zip(efficiency_summary.memory_mean, efficiency_summary.memory_sd)],
    "n": efficiency_summary.n,
})
efficiency_summary.to_csv(RUN_ROOT / "efficiency_mean_std.csv", index=False)
efficiency_table.to_csv(RUN_ROOT / "table_3_efficiency.csv", index=False)
(RUN_ROOT / "table_3_efficiency.tex").write_text(
    efficiency_table.drop(columns="n").to_latex(index=False, escape=False), encoding="utf-8"
)
display(efficiency_table)
print("Rate metrics exclude the first 10 warm-up steps; peak GiB is the busiest GPU's allocated memory.")